# Lesson 02 - Meneroka Rangka Kerja Ejen Microsoft

**Microsoft Agent Framework (MAF)** ialah rangka kerja sehenti untuk membina ejen AI. Ia menyediakan seni bina yang kemas dan boleh disusun dengan empat blok binaan teras:

- **Client** – menyambung ke titik akhir model AI dan mengendalikan komunikasi
- **Agent** – membungkus client dengan arahan dan definisi alat
- **Tools** – memperluaskan keupayaan ejen dengan fungsi tersuai yang boleh dipanggil oleh model
- **Session** – mengekalkan sejarah perbualan untuk interaksi berbilang giliran

Dalam pelajaran ini, kita akan membina **ejen tempahan perjalanan** yang memeriksa ketersediaan destinasi menggunakan konsep ini.


## Persediaan


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Memahami Seni Bina Framework Ejen

Microsoft Agent Framework mengikuti seni bina berlapis:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `AzureAIProjectAgentProvider` menyambung ke pelaksanaan Azure OpenAI. Ia mengendalikan pengesahan, pemformatan permintaan, dan pengecaman tindak balas.
2. **Agent** – Dibuat daripada klien melalui `provider.create_agent()`, ejen menggabungkan akses model dengan arahan (prompt sistem) dan alat.
3. **Tools** – Fungsi Python yang dihiasi dengan `@tool` yang boleh dipanggil oleh ejen untuk melaksanakan tindakan atau mendapatkan data.
4. **Session** – Objek `AgentSession` (dibuat melalui `agent.create_session()`) yang menyimpan sejarah perbualan, membolehkan dialog pelbagai giliran di mana ejen mengingati konteks sebelum ini.

Mari bina setiap lapisan langkah demi langkah.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Menambah Alat dengan Hiasan @tool

Alat membenarkan ejen melakukan tindakan selain menjana teks. Hiasan `@tool` menukar fungsi Python biasa kepada sesuatu yang boleh dipanggil oleh ejen.

Perkara penting:
- Gunakan `Annotated[type, "description"]` supaya model memahami setiap parameter.
- Docstring menjadi penerangan alat yang dilihat oleh model.
- `approval_mode="never_require"` bermaksud alat berjalan secara automatik tanpa pengesahan pengguna.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Membuat Ejen dengan Alat

Kini kita gabungkan klien, arahan, dan alat ke dalam satu ejen. `instructions` berfungsi sebagai prompt sistem — ia menentukan personaliti dan tingkah laku ejen tersebut.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Perbualan Berbilang Giliran dengan Sesi

Satu `AgentSession` (dicipta melalui `agent.create_session()`) menyimpan rekod semua mesej dalam satu perbualan. Dengan menggunakan sesi yang sama untuk setiap panggilan `agent.run()`, agen mempunyai akses kepada sejarah perbualan penuh dan boleh merujuk kembali ke mesej-mesej sebelumnya.

Kami menggunakan `tools=[check_destination_availability]` supaya agen boleh memanggil pemeriksa ketersediaan kami semasa setiap giliran.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Summary

Dalam pelajaran ini anda meneroka empat tiang Rangka Kerja Ejen Microsoft:

| Concept | What You Learned |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` berhubung dengan Azure OpenAI menggunakan pengesahan berasaskan kelayakan |
| **Agent** | `provider.create_agent()` menggabungkan sambungan model dengan arahan dan nama |
| **Tools** | Dekorator `@tool` mendedahkan fungsi Python untuk dipanggil oleh ejen |
| **Session** | `agent.create_session()` mengekalkan sejarah perbualan merentasi pelbagai giliran |

Blok binaan ini digabungkan untuk mencipta ejen yang boleh mengadakan perbualan semula jadi, memanggil fungsi luaran, dan mengekalkan konteks — asas untuk corak ejen yang lebih maju dalam pelajaran akan datang.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk mencapai ketepatan, sila maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya perlu dianggap sebagai sumber utama dan sahih. Bagi maklumat penting, terjemahan oleh penterjemah profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau tafsiran yang salah yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
